# API Flask para la Reference CNN — puerto 8080

Este notebook carga el `PRODUCTION_BUNDLE.zip` generado por el notebook ganador y publica una API local de inferencia. No entrena ni modifica el modelo. La API aplica la armonización y normalización serializadas antes de devolver la predicción Top-1 y el ranking Top-k.

Dependencias: `tensorflow`, `flask` y `numpy`. Por seguridad, el servidor escucha únicamente en `127.0.0.1`.

In [1]:
%pip install flask requests

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
from threading import Lock, Thread
import hashlib
import importlib.util
import json
import os
import zipfile

import numpy as np
import tensorflow as tf
from flask import Flask, jsonify, request
from werkzeug.serving import make_server

print("TensorFlow:", tf.__version__)
print("✓ Dependencias cargadas")

TensorFlow: 2.21.0
✓ Dependencias cargadas


## 1. Configuración

Si el ZIP está en otra ubicación, cambia `BUNDLE_PATH`. También puede definirse la variable de entorno `REFERENCE_CNN_BUNDLE`.

In [3]:
DEFAULT_BUNDLE_PATH = Path(
    "raman_all_7_zips_mixed_fixed_split_5_seeds/results/PRODUCTION_BUNDLE.zip"
)
BUNDLE_PATH = Path(os.environ.get("REFERENCE_CNN_BUNDLE", DEFAULT_BUNDLE_PATH)).expanduser()
EXTRACT_DIR = Path("reference_cnn_api_artifacts")
HOST = "127.0.0.1"
PORT = 8081
MAX_TOP_K = 10

print("Paquete esperado:", BUNDLE_PATH.resolve())
print(f"API: http://{HOST}:{PORT}")

Paquete esperado: /Users/nico/anaconda_projects/TFM - FINAL/raman_all_7_zips_mixed_fixed_split_5_seeds/results/PRODUCTION_BUNDLE.zip
API: http://127.0.0.1:8081


## 2. Carga y verificación del paquete

Se comprueban los archivos obligatorios y sus hashes SHA-256 antes de cargar el modelo.

In [4]:
if not BUNDLE_PATH.is_file():
    raise FileNotFoundError(
        f"No se encontró {BUNDLE_PATH}. Ejecuta primero la sección de exportación "
        "del notebook ganador o corrige BUNDLE_PATH."
    )

EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(BUNDLE_PATH, "r") as archive:
    archive.extractall(EXTRACT_DIR)

required_files = {
    "production_model.keras",
    "class_labels.json",
    "raman_grid.npy",
    "preprocessing_contract.json",
    "preprocessing_reference.py",
    "production_selection.json",
    "artifact_manifest.json",
}
missing_files = sorted(name for name in required_files if not (EXTRACT_DIR / name).is_file())
if missing_files:
    raise RuntimeError(f"El paquete está incompleto. Faltan: {missing_files}")

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as file_handle:
        for chunk in iter(lambda: file_handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

artifact_manifest = json.loads((EXTRACT_DIR / "artifact_manifest.json").read_text(encoding="utf-8"))
for item in artifact_manifest["files"]:
    artifact_path = EXTRACT_DIR / item["name"]
    if not artifact_path.is_file():
        raise RuntimeError(f"Falta el artefacto declarado: {item['name']}")
    if artifact_path.stat().st_size != int(item["size_bytes"]):
        raise RuntimeError(f"Tamaño inválido: {item['name']}")
    if sha256_file(artifact_path) != item["sha256"]:
        raise RuntimeError(f"Hash SHA-256 inválido: {item['name']}")

print("✓ Paquete íntegro y verificado")

✓ Paquete íntegro y verificado


In [5]:
labels = json.loads((EXTRACT_DIR / "class_labels.json").read_text(encoding="utf-8"))
contract = json.loads((EXTRACT_DIR / "preprocessing_contract.json").read_text(encoding="utf-8"))
selection = json.loads((EXTRACT_DIR / "production_selection.json").read_text(encoding="utf-8"))

module_spec = importlib.util.spec_from_file_location(
    "reference_cnn_preprocessing", EXTRACT_DIR / "preprocessing_reference.py"
)
preprocessing_module = importlib.util.module_from_spec(module_spec)
module_spec.loader.exec_module(preprocessing_module)
raman_grid = preprocessing_module.load_raman_grid(EXTRACT_DIR)
preprocess_raw_spectrum = preprocessing_module.preprocess_raw_spectrum

model = tf.keras.models.load_model(EXTRACT_DIR / "production_model.keras", compile=False)
n_classes = len(labels)
expected_input_shape = (int(contract["raman_grid_points"]), 1)
if tuple(model.input_shape[1:]) != expected_input_shape:
    raise RuntimeError(f"Forma de entrada incompatible: {model.input_shape}")
if int(model.output_shape[-1]) != n_classes:
    raise RuntimeError("El número de salidas del modelo no coincide con las etiquetas.")

prediction_lock = Lock()
print(f"✓ Modelo cargado: entrada {model.input_shape}; {n_classes} clases")
print("Semilla seleccionada:", selection.get("selected_seed"))

✓ Modelo cargado: entrada (None, 512, 1); 255 clases
Semilla seleccionada: 123


## 3. API de inferencia

`POST /predict` espera `raman_shift_cm-1`, `intensity` y, opcionalmente, `top_k`. El servidor no aplica aumento de datos durante la inferencia.

In [6]:
app = Flask("reference_cnn_raman_api")

@app.get("/health")
def health():
    return jsonify({
        "status": "ok",
        "service": "reference-cnn-raman",
        "n_classes": n_classes,
        "input_points": int(raman_grid.size),
        "raman_lower_cm-1": float(raman_grid[0]),
        "raman_upper_cm-1": float(raman_grid[-1]),
        "selected_seed": selection.get("selected_seed"),
    })

@app.post("/predict")
def predict():
    payload = request.get_json(silent=True)
    if not isinstance(payload, dict):
        return jsonify({"error": "Se esperaba un objeto JSON."}), 400

    if "raman_shift_cm-1" not in payload or "intensity" not in payload:
        return jsonify({
            "error": "Faltan raman_shift_cm-1 o intensity."
        }), 400

    try:
        top_k = int(payload.get("top_k", 5))
        if not 1 <= top_k <= min(MAX_TOP_K, n_classes):
            raise ValueError(f"top_k debe estar entre 1 y {min(MAX_TOP_K, n_classes)}.")
        model_input = preprocess_raw_spectrum(
            payload["raman_shift_cm-1"], payload["intensity"], raman_grid
        )
        with prediction_lock:
            probabilities = model.predict(model_input, verbose=0)[0]
        if probabilities.shape != (n_classes,) or not np.isfinite(probabilities).all():
            raise RuntimeError("El modelo devolvió probabilidades inválidas.")
    except (TypeError, ValueError) as exc:
        return jsonify({"error": str(exc)}), 400
    except Exception as exc:
        app.logger.exception("Error de inferencia")
        return jsonify({"error": "Error interno durante la inferencia."}), 500

    ranked_indices = np.argsort(probabilities)[::-1][:top_k]
    ranking = [
        {
            "rank": rank,
            "class_index": int(class_index),
            "label": str(labels[class_index]),
            "probability": float(probabilities[class_index]),
        }
        for rank, class_index in enumerate(ranked_indices, start=1)
    ]
    return jsonify({
        "predicted_class": ranking[0]["label"],
        "confidence": ranking[0]["probability"],
        "top_k": ranking,
        "processed_shape": list(model_input.shape),
        "warning": (
            "Clasificador de conjunto cerrado: siempre propone una clase conocida. "
            "La confianza softmax no demuestra que la muestra pertenezca a RRUFF."
        ),
    })

print("✓ Endpoints /health y /predict preparados")

✓ Endpoints /health y /predict preparados


## 4. Iniciar el servidor

El servidor se ejecuta en un hilo para que Jupyter siga disponible. Si el puerto 8080 está ocupado por otro proceso, esta celda informa el error y no elimina procesos ajenos.

In [7]:
if "api_server" in globals():
    try:
        api_server.shutdown()
    except Exception:
        pass

try:
    api_server = make_server(HOST, PORT, app, threaded=True)
except OSError as exc:
    raise RuntimeError(
        f"No fue posible abrir {HOST}:{PORT}. Comprueba qué proceso utiliza el puerto."
    ) from exc

api_thread = Thread(target=api_server.serve_forever, daemon=True)
api_thread.start()
print(f"✓ API activa en http://{HOST}:{PORT}")
print("Para detenerla cuando termines, ejecuta: api_server.shutdown()")

✓ API activa en http://127.0.0.1:8081
Para detenerla cuando termines, ejecuta: api_server.shutdown()


In [9]:
import requests

response = requests.get("http://127.0.0.1:8081/health", timeout=5)
print(response.status_code)
print(response.json())

127.0.0.1 - - [08/Sep/2026 14:07:20] "GET /health HTTP/1.1" 200 -


200
{'input_points': 512, 'n_classes': 255, 'raman_lower_cm-1': 180.70899963378906, 'raman_upper_cm-1': 1236.155029296875, 'selected_seed': 123, 'service': 'reference-cnn-raman', 'status': 'ok'}


127.0.0.1 - - [08/Sep/2026 14:07:35] "GET /health HTTP/1.1" 200 -
127.0.0.1 - - [08/Sep/2026 14:07:51] "POST /predict HTTP/1.1" 200 -
127.0.0.1 - - [08/Sep/2026 14:16:10] "GET /health HTTP/1.1" 200 -
127.0.0.1 - - [08/Sep/2026 14:16:10] "POST /predict HTTP/1.1" 200 -


Mantén este kernel activo mientras ejecutas el notebook cliente. Si el cliente está en otro equipo, será necesaria una red o un túnel seguros; este prototipo local no incorpora autenticación, HTTPS ni detección de muestras fuera de distribución.